# Overview — paper-style summary across all (pair, dataset) notebooks

Scans every `evaluation_results/{old,improved}/` directory under
`src/interpretability/improved_pipeline/<pair>/<dataset>/`, then prints one row
per (pair, dataset, side) with:

- **next_event_accuracy** — first-event argmax matches actual
- **dl_similarity_proba** — mean DL similarity (1 − normalized DL distance) over the 100 stochastic suffix samples, then averaged over prefixes
- **dl_similarity_argmax** — same metric on the argmax (`mean`) prediction; depressed when the decoder rarely emits EOS

Activity key per dataset is read from `src/interpretability/config/<dataset>_config.py`.

Re-run this whenever you finish sampling a new pair — it just reads chunk files.

In [1]:
import os, sys, glob, pickle, re
from pathlib import Path

_REPO_ROOT = Path('.').resolve()
while not (_REPO_ROOT / 'src').is_dir() and _REPO_ROOT.parent != _REPO_ROOT:
    _REPO_ROOT = _REPO_ROOT.parent
sys.path.insert(0, str(_REPO_ROOT))
sys.path.insert(0, str(_REPO_ROOT / 'src'))

from src.evaluation_metrics.metrics import NormalizedDamerauLevenshteinDistanceMean

PIPELINE_ROOT = _REPO_ROOT / 'src/interpretability/improved_pipeline'
CONFIG_DIR    = _REPO_ROOT / 'src/interpretability/config'

def activity_key_for(dataset: str) -> str:
    """Pull `concept_name` out of `<dataset>_config.py`."""
    cfg = CONFIG_DIR / f'{dataset}_config.py'
    if not cfg.exists():
        return 'concept:name'
    m = re.search(r"concept_name\s*=\s*['\"]([^'\"]+)['\"]", cfg.read_text())
    return m.group(1) if m else 'concept:name'

def evaluate_chunks(chunk_dir: Path, attr: str):
    """Evaluate the saved sampling chunks. Empty mean predictions (model
    immediately predicting EOS) count as wrong for next-event accuracy and
    contribute 0 similarity for the argmax DL — they're tracked separately
    via `n_empty_argmax` so total mode collapse is visible."""
    files = sorted(chunk_dir.glob('results_part_*.pkl'))
    if not files:
        return None
    dl = NormalizedDamerauLevenshteinDistanceMean(attr)
    n_total = n_correct = n_empty_argmax = 0
    sim_argmax_sum = sim_proba_sum = 0.0
    for f in files:
        with open(f, 'rb') as fh:
            chunk = pickle.load(fh)
        for _, (pref, suf, mp, pp) in chunk.items():
            if not suf:
                continue
            n_total += 1
            if mp and attr in mp[0]:
                n_correct += int(mp[0][attr] == suf[0][attr])
            else:
                n_empty_argmax += 1
            r = dl.evaluate(pref, suf, mp, pp)
            sim_argmax_sum += r['mean']
            sim_proba_sum  += r['prob']
    if n_total == 0:
        return None
    return dict(n_prefixes           = n_total,
                next_event_accuracy  = n_correct / n_total,
                dl_similarity_proba  = sim_proba_sum / n_total,
                dl_similarity_argmax = sim_argmax_sum / n_total,
                n_empty_argmax       = n_empty_argmax)

In [2]:
rows = []
for pair_dir in sorted(PIPELINE_ROOT.glob('*/')):
    pair = pair_dir.name
    if pair == '_shared':
        continue
    for ds_dir in sorted(pair_dir.glob('*/')):
        if not (ds_dir / 'evaluation_results').is_dir():
            continue
        attr = activity_key_for(ds_dir.name)
        for side in ('old', 'improved'):
            sd = ds_dir / 'evaluation_results' / side
            if not sd.is_dir() or not list(sd.glob('results_part_*.pkl')):
                rows.append(dict(pair=pair, dataset=ds_dir.name, side=side,
                                 activity_key=attr, n_prefixes=None,
                                 next_event_accuracy=None,
                                 dl_similarity_proba=None,
                                 dl_similarity_argmax=None))
                continue
            res = evaluate_chunks(sd, attr)
            if res is None:
                rows.append(dict(pair=pair, dataset=ds_dir.name, side=side,
                                 activity_key=attr, n_prefixes=None,
                                 next_event_accuracy=None,
                                 dl_similarity_proba=None,
                                 dl_similarity_argmax=None))
                continue
            rows.append(dict(pair=pair, dataset=ds_dir.name, side=side,
                             activity_key=attr, **res))

import pandas as pd
df = pd.DataFrame(rows)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if pd.notnull(x) else '—')
df

,pair,dataset,side,activity_key,n_prefixes,next_event_accuracy,dl_similarity_proba,dl_similarity_argmax,n_empty_argmax
0,camargo,bpic17,old,concept:name,3988.0000,0.8992,0.3314,0.1476,20.0000
1,camargo,bpic17,improved,concept:name,NaN,NaN,NaN,NaN,NaN
2,camargo,domestic_declarations,old,Activity,9171.0000,0.8708,0.8809,0.9243,1.0000
3,camargo,domestic_declarations,improved,Activity,NaN,NaN,NaN,NaN,NaN
4,camargo,helpdesk,old,Activity,3329.0000,0.8044,0.7275,0.8597,10.0000
5,camargo,helpdesk,improved,Activity,NaN,NaN,NaN,NaN,NaN
6,camargo,sepsis,old,concept:name,2604.0000,0.6494,0.2061,0.1722,44.0000
7,camargo,sepsis,improved,concept:name,2604.0000,0.6321,0.2062,0.1836,57.0000
8,henryk,bpic17,old,concept:name,9639.0000,0.8192,0.2911,0.3043,69.0000
9,henryk,bpic17,improved,concept:name,9639.0000,0.6822,0.2518,0.2619,60.0000
